# Навчання сегментації U-Net

Етап 6. Ноутбук проходить від початку до кінця і локально (CPU, `data/train_micro/` — перевірка логіки),
і на Kaggle (GPU — справжнє навчання). Структура — як у `03_classifier_train.ipynb`.

**На Kaggle:**
1. Settings → Accelerator → **GPU**, Internet → **On**; секрет `KAGGLE_API_TOKEN` прикріплений;
   Input — `airbus-ship-detection-src` останньої версії (з `ShipSegmentationDataset` у `dataset.py`).
2. Пробний запуск: `PILOT = True` (клітинка налаштувань у розділі 1) → **Run All** в інтерактивній сесії.
3. Повне навчання: `PILOT = False` → **Save Version → Save & Run All**.
   Результати — у вкладці **Output** збереженої версії.

## 0. Дані на Kaggle

Локально цей розділ нічого не робить. На Kaggle: токен із секретів, бібліотеки → завантаження архіву
змагання (~5 хв) → розпакування лише `train_v2/` і CSV. Якщо дані вже розпаковані в цій сесії — крок пропускається.

In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
KAGGLE_DATA_DIR = Path("/kaggle/tmp/data")

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    # Встановлена в Kaggle версія kaggle може не знати нового формату токена.
    !pip install -q kaggle==2.2.4 segmentation-models-pytorch==0.5.0

In [ ]:
if ON_KAGGLE and not (KAGGLE_DATA_DIR / "train_v2").exists():
    !mkdir -p /kaggle/tmp && cd /kaggle/tmp && kaggle competitions download -c airbus-ship-detection
    !unzip -q /kaggle/tmp/airbus-ship-detection.zip "train_v2/*" train_ship_segmentations_v2.csv -d {KAGGLE_DATA_DIR}

if ON_KAGGLE:
    n_files = sum(1 for _ in (KAGGLE_DATA_DIR / "train_v2").iterdir())
    print("знімків у train_v2:", n_files)
    assert n_files == 192556

## 1. Модель і цикл навчання

| | Локально (CPU) | Kaggle, `PILOT = True` | Kaggle, `PILOT = False` |
|---|---|---|---|
| знімки | `data/train_micro/`, лише з кораблями | `/kaggle/tmp/data/train_v2/` | те саме |
| train | ~120 знімків, 2 батчі | ~38 300 знімків, 50 батчів | ~38 300, усі |
| val | ~30 знімків | 1 000 випадкових з кораблями | усі 4 256 з кораблями |
| епох | 1 | 1 | 8 |
| результати | `data/unet_local/` | `/kaggle/working/` | `/kaggle/working/` |

Train і val — **лише знімки з кораблями**: порожні відсіює класифікатор. Повний пайплайн на всіх
val-знімках — на етапі 8.

In [ ]:
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
from torch.utils.data import DataLoader

PILOT = True

if ON_KAGGLE:
    sys.path.append("/kaggle/input/datasets/yurapolishchuk/airbus-ship-detection-src")
    DATA_DIR = KAGGLE_DATA_DIR
    IMAGES_DIR = DATA_DIR / "train_v2"
    OUT_DIR = Path("/kaggle/working")
    DEVICE = "cuda"
    MAX_TRAIN_BATCHES = 50 if PILOT else None
    MAX_VAL_IMAGES = 1000 if PILOT else None
    EPOCHS = 1 if PILOT else 8
    NUM_WORKERS = 4
    BATCH_SIZE = 16
else:
    ROOT = Path.cwd().parent
    sys.path.append(str(ROOT / "src"))
    DATA_DIR = ROOT / "data"
    IMAGES_DIR = DATA_DIR / "train_micro"
    OUT_DIR = DATA_DIR / "unet_local"
    DEVICE = "cpu"
    MAX_TRAIN_BATCHES = 2
    MAX_VAL_IMAGES = None
    EPOCHS = 1
    NUM_WORKERS = 0
    BATCH_SIZE = 4

from dataset import IMAGENET_MEAN, IMAGENET_STD, ShipSegmentationDataset, rles_by_image
from split import make_split

SEED = 42
LR = 3e-4
USE_AMP = DEVICE == "cuda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Kaggle:", ON_KAGGLE, "| PILOT:", PILOT, "| device:", DEVICE, "| AMP:", USE_AMP)

### 1.1. Дані

Поділ — тією самою `make_split` (seed 42 → той самий поділ, що в класифікатора).
Локально `make_split` ділить лише 300 знімків `train_micro` (80/20).
Словник `ImageId → список RLE` будуємо один раз — Dataset не шукає рядки в CSV при кожному зверненні.

In [ ]:
masks = pd.read_csv(DATA_DIR / "train_ship_segmentations_v2.csv")
rles = rles_by_image(masks)
if ON_KAGGLE:
    split = make_split(masks)
else:
    # З 300 знімків train_micro у справжню val потрапляє надто мало — ділимо саму вибірку.
    available = {p.name for p in IMAGES_DIR.iterdir()}
    split = make_split(masks[masks["ImageId"].isin(available)], val_size=0.2)

with_ships = split[split["n_ships"] > 0]
train_df = with_ships[with_ships["split"] == "train"].reset_index(drop=True)
val_df = with_ships[with_ships["split"] == "val"].reset_index(drop=True)
n_val_full = len(val_df)
if MAX_VAL_IMAGES is not None:
    val_df = val_df.sample(n=MAX_VAL_IMAGES, random_state=SEED).reset_index(drop=True)

train_dataset = ShipSegmentationDataset(train_df["ImageId"], rles, IMAGES_DIR, train=True)
val_dataset = ShipSegmentationDataset(val_df["ImageId"], rles, IMAGES_DIR, train=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"train: {len(train_df)} знімків з кораблями")
print(f"val:   {len(val_df)} знімків з кораблями")

### 1.2. Перевірка `ShipSegmentationDataset` очима

4 знімки з train **після аугментацій** (як їх бачить модель) з напівпрозорою маскою. Корабель має бути
під маскою — тоді знімок і маска повертаються однаково. Тензор знімка «рознормалізуємо» назад для показу.

In [ ]:
def to_display(image_tensor):
    image = image_tensor.numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(image, 0, 1)


def overlay(image, mask, color=(1.0, 0.0, 0.0)):
    shown = image.copy()
    shown[mask > 0.5] = 0.6 * shown[mask > 0.5] + 0.4 * np.array(color)
    return shown


fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, i in zip(axes, range(4)):
    image, mask = train_dataset[i]
    print(f"{train_df['ImageId'][i]}: знімок {tuple(image.shape)}, маска {tuple(mask.shape)}, "
          f"значення маски {torch.unique(mask).tolist()}, пікселів корабля {int(mask.sum())}")
    ax.imshow(overlay(to_display(image), mask[0].numpy()))
    ax.set_title(train_df["ImageId"][i])
    ax.axis("off")
plt.tight_layout()
plt.show()

### 1.3. Модель і loss

`smp.Unet` — encoder ResNet34 з вагами ImageNet (той самий, що в класифікаторі, без останнього шару),
decoder з 5 блоків «збільшити вдвічі → склеїти з картою ознак encoder → 2 згортки» і голова — згортка 1×1,
що дає 1 логіт на піксель. Pretrained лише encoder.

Loss = **BCE + Dice**: BCE дає стабільний градієнт кожному пікселю, Dice — стійкість до дисбалансу пікселів
(корабель — соті частки відсотка знімка).

In [ ]:
model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", classes=1).to(DEVICE)

bce_loss = torch.nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)


def loss_fn(logits, masks):
    return bce_loss(logits, masks) + dice_loss(logits, masks)


optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

print(f"ваг у моделі: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} млн")

### 1.4. Метрика: Dice по знімках

Для кожного знімка: p > 0.5 → бінарна маска → Dice з правильною → середнє по знімках.
Саме по знімках, а не по всіх пікселях разом: інакше великі кораблі «заглушили» б дрібні
(а метрика змагання теж рахується по знімках). Якщо обидві маски порожні — Dice = 1.

Спершу перевіримо функцію на прикладах з відомою відповіддю.

In [ ]:
def dice_per_image(probs, masks, threshold=0.5):
    predicted = (probs > threshold).float()
    intersection = (predicted * masks).sum(dim=(1, 2, 3))
    total = predicted.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
    return torch.where(total > 0, 2 * intersection / total.clamp(min=1), torch.ones_like(total))


truth = torch.zeros(1, 1, 4, 4)
truth[0, 0, :2, :2] = 1
half = torch.zeros(1, 1, 4, 4)
half[0, 0, :2, :1] = 1
checks = {
    "ідеальна маска": (truth, 1.0),
    "порожнє передбачення": (torch.zeros(1, 1, 4, 4), 0.0),
    "половина корабля": (half, 2 * 2 / (2 + 4)),
}
for name, (prediction, expected) in checks.items():
    result = dice_per_image(prediction, truth).item()
    assert np.isclose(result, expected), name
    print(f"OK  {name}: {result:.4f}")
print("OK  обидві порожні:", dice_per_image(torch.zeros(1, 1, 4, 4), torch.zeros(1, 1, 4, 4)).item())

### 1.5. Навчання однієї епохи і оцінка

Ті самі 5 кроків на батч, що в класифікаторі; змінились лише форма виходу (`[B, 1, 384, 384]` замість `[B]`)
і loss. `evaluate` повертає val loss і Dice для кожного знімка.

In [ ]:
def train_one_epoch(model, loader, max_batches=None):
    model.train()
    losses = []
    for i, (images, masks) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = model(images)
        loss = loss_fn(logits.float(), masks)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
    return np.mean(losses)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    losses, dices = [], []
    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = model(images)
        losses.append(loss_fn(logits.float(), masks).item())
        dices.append(dice_per_image(torch.sigmoid(logits.float()), masks).cpu())
    return np.mean(losses), torch.cat(dices).numpy()

### 1.6. Цикл по епохах

Після кожної епохи: train loss, val loss, **val Dice** (середнє по знімках), час. Якщо Dice кращий
за всі попередні епохи — зберігаємо ваги (`unet_resnet34.pth`) і Dice кожного val-знімка (`val_dice.csv`).
У пілотному запуску — ще й швидкість і оцінка часу повної епохи.

In [ ]:
best_dice = -1
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, MAX_TRAIN_BATCHES)
    train_seconds = time.time() - start
    val_loss, val_dices = evaluate(model, val_loader)
    val_seconds = time.time() - start - train_seconds
    mean_dice = val_dices.mean()
    minutes = (time.time() - start) / 60
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_dice": mean_dice,
                    "minutes": minutes})
    print(f"епоха {epoch}: train loss {train_loss:.4f} | val loss {val_loss:.4f} | val Dice {mean_dice:.4f} | {minutes:.1f} хв")

    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save(model.state_dict(), OUT_DIR / "unet_resnet34.pth")
        val_df.assign(dice=val_dices)[["ImageId", "n_ships", "dice"]].to_csv(OUT_DIR / "val_dice.csv", index=False)
        print("    ↑ найкраща епоха — ваги і val_dice.csv збережено")

    if PILOT:
        n_train_seen = min(MAX_TRAIN_BATCHES or len(train_loader), len(train_loader)) * BATCH_SIZE
        train_speed = n_train_seen / train_seconds
        val_speed = len(val_df) / val_seconds
        full_epoch = len(train_df) / train_speed + n_val_full / val_speed
        print(f"    train: {train_speed:.0f} знімків/с, val: {val_speed:.0f} знімків/с")
        print(f"    оцінка повної епохи: {full_epoch / 60:.0f} хв "
              f"({len(train_df)} train + {n_val_full} val знімків)")

pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)

### 1.7. Приклади передбачень і перевірка збережених ваг

Перезавантажуємо збережені ваги в нову модель і показуємо 4 val-знімки: знімок, правда (червоне),
передбачення p > 0.5 (зелене). Картинка зберігається в `val_examples.png`.
Dice перезавантаженої моделі має збігатися зі збереженим.

In [ ]:
reloaded = smp.Unet(encoder_name="resnet34", encoder_weights=None, classes=1)
reloaded.load_state_dict(torch.load(OUT_DIR / "unet_resnet34.pth", map_location=DEVICE))
reloaded = reloaded.to(DEVICE).eval()

_, reloaded_dices = evaluate(reloaded, val_loader)
saved = pd.read_csv(OUT_DIR / "val_dice.csv")
print("макс. різниця Dice після перезавантаження ваг:", np.abs(reloaded_dices - saved["dice"].values).max())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col in range(4):
    image, mask = val_dataset[col]
    with torch.no_grad():
        prob = torch.sigmoid(reloaded(image.unsqueeze(0).to(DEVICE))).cpu()[0, 0].numpy()
    shown = to_display(image)
    axes[0, col].imshow(overlay(shown, mask[0].numpy()))
    axes[0, col].set_title(f"правда: {val_df['ImageId'][col]}")
    axes[1, col].imshow(overlay(shown, prob, color=(0.0, 1.0, 0.0)))
    axes[1, col].set_title(f"передбачення, Dice {saved['dice'][col]:.2f}")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "val_examples.png", dpi=80)
plt.show()